In [1]:
import sys
!{sys.executable} -m pip install -U google-genai python-dotenv ipywidgets
from dotenv import load_dotenv
import os
load_dotenv() 
gemini_key = os.getenv("GEMINI_API_KEY")
print(gemini_key[:6])

AIzaSy



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# 1) 후보를 "오른쪽"에 가로 배치하려면 RadioButtons 대신 ToggleButtons 사용이 가장 깔끔합니다.
# 2) 줄(테두리/구분선) 제거 + 3) 글자색 검정으로 강제: CSS 오버라이드

display(HTML("""
<style>
/* 라벨/텍스트 색상: 검정 */
.widget-label, .jupyter-widgets label, .widget-inline-hbox, .widget-inline-vbox,
.jupyter-widgets .widget-html-content, .jupyter-widgets .widget-text input,
.jupyter-widgets .widget-textarea textarea {
  color: #000 !important;
}

/* ToggleButtons 글자/테두리 정리 */
.jupyter-widgets .widget-toggle-button .btn,
.jupyter-widgets .widget-toggle-buttons .btn {
  color: #000 !important;
  border: none !important;      /* 줄(테두리) 제거 */
  box-shadow: none !important;  /* 줄(그림자) 제거 */
}

/* 선택된 버튼도 테두리/그림자 제거 */
.jupyter-widgets .widget-toggle-button .btn.active,
.jupyter-widgets .widget-toggle-buttons .btn.active {
  border: none !important;
  box-shadow: none !important;
}

/* 포커스 아웃라인 제거(원하면 주석 처리 가능) */
.jupyter-widgets .btn:focus {
  outline: none !important;
  box-shadow: none !important;
}
</style>
"""))

LABEL_W = "80px"
FIELD_W = "700px"

common_style = {"description_width": LABEL_W}

topic_w = widgets.Text(description="보고서 제목", layout=widgets.Layout(width="300px"), style=common_style)
purpose_w = widgets.Text(description="보고서 목적", layout=widgets.Layout(width="600px"), style=common_style)

# 후보(옵션) 오른쪽 배치: Label + ToggleButtons 를 HBox로 구성
structure_tb = widgets.ToggleButtons(
    options=[("줄글형", "줄글형"), ("Bullet 항목형", "Bullet 항목형")],
    value="줄글형",
    layout=widgets.Layout(width="360px"),
    style={"button_width": "120px"}
)
structure_row = widgets.HBox(
    [widgets.Label("서술 구조", layout=widgets.Layout(width=LABEL_W)), structure_tb],
    layout=widgets.Layout(width="600px")
)

gemini_tb = widgets.ToggleButtons(
    options=[("Flash", "flash"), ("Pro", "pro")],
    value="flash",
    layout=widgets.Layout(width="360px"),
    style={"button_width": "120px"}
)
gemini_row = widgets.HBox(
    [widgets.Label("Gemini 모델", layout=widgets.Layout(width=LABEL_W)), gemini_tb],
    layout=widgets.Layout(width="600px")
)

requirements_w = widgets.Textarea(
    description="요구사항",
    layout=widgets.Layout(width="600px", height="100px"),
    placeholder="예: 1. 2.",
    style=common_style
)

btn = widgets.Button(description="확인", button_style="primary")
btn_box = widgets.HBox([btn], layout=widgets.Layout(justify_content="center", width="700px"))
out = widgets.Output()

result = {}

def on_click(_):
    result["topic"] = topic_w.value
    result["purpose"] = purpose_w.value
    result["structure"] = structure_tb.value
    result["gemini_model"] = gemini_tb.value
    result["requirements"] = requirements_w.value
    with out:
        clear_output()
        print("입력 완료")
        print(result)

btn.on_click(on_click)

display(topic_w, purpose_w, structure_row, gemini_row, requirements_w, btn_box, out)

Text(value='', description='보고서 제목', layout=Layout(width='300px'), style=TextStyle(description_width='80px'))

Text(value='', description='보고서 목적', layout=Layout(width='600px'), style=TextStyle(description_width='80px'))

Textarea(value='', description='요구사항', layout=Layout(height='100px', width='600px'), placeholder='예: 1. 2.', s…

Output()

In [4]:
# 목차 생성

import os
from google import genai
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

# 상태 저장
toc_state = {"toc": "", "final_toc": ""}

def get_inputs(default_topic="미정(보고서제목)", default_purpose="미정(보고서목적)", default_requirements="없음"):
    r = globals().get("result", {}) or {}

    def pick(key, widget_name):
        w = globals().get(widget_name, None)
        return (r.get(key) or (getattr(w, "value", "") if w else "") or "").strip()

    topic = pick("topic", "topic_w") or default_topic
    purpose = pick("purpose", "purpose_w") or default_purpose
    requirements = pick("requirements", "requirements_w") or default_requirements
    return topic, purpose, requirements

def make_prompt(mode, topic, purpose, requirements, toc=None, feedback=None):
    """mode: 'toc' | 'revise'"""
    if mode == "toc":
        return f"""
너는 컨설팅 보고서 작성 전문가다.
아래 입력을 바탕으로 '보고서 목차(TOC)'를 한국어로 작성하라.

[입력]
- 보고서 제목: {topic}
- 보고서 목적: {purpose}
- 조건/요구사항: {requirements}

[출력 요구사항]
- 출력은 번호가 매겨진 목차 항목만 포함해야 한다.
- 제목, 설명, 안내문, 라벨을 절대 출력하지 말 것.
- "목차", "개정 목차", "개정목차"라는 문자열을 어떤 형태로도 출력하지 말 것.
- #, ##, ###, *, -, • 등 마크다운 기호를 절대 사용하지 말 것.
- 번호 체계는 1, 1.1, 1.2 형태로 작성
- 최소 5개 장(Chapter) 이상
- 장(1,2,3...)과 소절(1.1, 1.2...)로 구성
- 불필요한 설명문 없이 목차만 출력
- 도입부에 목차라는 표시 표현 생성 금지
- # 또는 * 절대 사용금지
""".strip()

    if mode == "revise":
        return f"""
너는 컨설팅 보고서 편집자다.
아래 기존 목차와 사용자 수정 지시를 반영하여 '개정 목차'를 작성하라.

[입력]
- 보고서 제목: {topic}
- 보고서 목적: {purpose}
- 조건/요구사항: {requirements}

[기존 목차]
{toc}

[목차 수정]
{feedback}

[출력 요구사항]
- 출력은 번호가 매겨진 목차 항목만 포함해야 한다.
- 제목, 설명, 안내문, 라벨을 절대 출력하지 말 것.
- "목차", "개정 목차", "개정목차"라는 문자열을 어떤 형태로도 출력하지 말 것.
- #, ##, ###, *, -, • 등 마크다운 기호를 절대 사용하지 말 것.
- 번호 체계는 1, 1.1, 1.2 형태로 작성
- 불필요한 설명문 없이 목차만 출력
- 도입부에 개정목차라는 표시 표현 생성 금지
- # 또는 * 기호 절대 사용하지 말것
""".strip()

    raise ValueError("mode는 'toc' 또는 'revise'여야 합니다.")

def toc_ui(model_name="gemini-2.0-flash", width="800px"):
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise RuntimeError("GEMINI_API_KEY 환경변수가 없습니다. (.env 로드 또는 OS 환경변수 설정 필요)")

    topic, purpose, requirements = get_inputs()
    client = genai.Client(api_key=api_key)

    # 최초 목차 생성
    resp = client.models.generate_content(
        model=model_name,
        contents=make_prompt("toc", topic, purpose, requirements)
    )
    toc_state["toc"] = (resp.text or "").strip()
    toc_state["final_toc"] = toc_state["toc"]

    out = widgets.Output()

    def render():
        with out:
            clear_output()
            display(Markdown(" 생성된 목차\n\n```text\n" + toc_state["final_toc"] + "\n```"))

    feedback_w = widgets.Textarea(
        description="목차 수정",
        placeholder="추가, 수정, 삭제 가능합니다.",
        style={"description_width": "80px"},
        layout=widgets.Layout(width=width, height="120px")
    )

    apply_btn = widgets.Button(description="반영", button_style="primary")
    btn_box = widgets.HBox([apply_btn], layout=widgets.Layout(justify_content="center", width=width))

    def on_apply(_):
        feedback = (feedback_w.value or "").strip()
        if not feedback:
            toc_state["final_toc"] = toc_state["toc"]
        else:
            r = client.models.generate_content(
                model=model_name,
                contents=make_prompt("revise", topic, purpose, requirements, toc=toc_state["toc"], feedback=feedback)
            )
            toc_state["final_toc"] = (r.text or "").strip()

        # 최종 목차를 화면에 반영
        render()

    apply_btn.on_click(on_apply)

    render()
    display(out, feedback_w, btn_box)

# 실행
toc_ui(model_name="gemini-2.0-flash")


Output()

Textarea(value='', description='목차 수정', layout=Layout(height='120px', width='800px'), placeholder='추가, 수정, 삭제 …

In [5]:
from docx import Document
import os
import re
import time
from google import genai

# =========================
# 1) 유틸: 파일명/경로/문서 기록
# =========================
def sanitize_filename(name: str, default="report"):
    name = (name or "").strip()
    if not name:
        return default
    name = re.sub(r'[\\/:*?"<>|]+', "_", name)   # Windows 금지 문자 치환
    name = re.sub(r"\s+", " ", name).strip()     # 공백 정리
    return name[:120]

def get_download_path(file_name="report.docx"):
    if os.name == "nt":  # Windows
        download_path = os.path.join(os.environ.get("USERPROFILE", ""), "Downloads")
    else:  # macOS, Linux
        download_path = os.path.join(os.environ.get("HOME", ""), "Downloads")

    if not download_path or not os.path.isdir(download_path):
        download_path = os.getcwd()

    os.makedirs(download_path, exist_ok=True)
    return os.path.join(download_path, file_name)

def add_multiline_text(doc: Document, text: str):
    for line in (text or "").splitlines():
        if line.strip() == "":
            doc.add_paragraph("")  # 빈 줄 유지
        else:
            doc.add_paragraph(line)

# =========================
# 2) TOC 파싱/프롬프트/블록 파싱(기존 로직 유지)
# =========================
def extract_chapters(final_toc: str):
    chapters = []
    pattern = re.compile(r"^\s*(\d+(?:\.\d+)*)\s*[\.\)]?\s+.+?:?\s*$")
    for line in (final_toc or "").splitlines():
        s = line.strip().replace("*", "")
        if pattern.match(s):
            chapters.append(s.rstrip(":").strip())
    return chapters

def make_prompt_for_report(chapter_title, topic, purpose, requirements, summary_context=""):
    return f"""
너는 대형 컨설팅펌 수준의 전문 보고서를 작성하는 최고 수준의 분석가이다.
아래 입력 정보를 바탕으로, 해당 소제목에 대해서
심층적이고 장문(긴 분량)의 고품질 보고서를 작성하라.

[입력]
- 해당 소제목: {chapter_title}
- 보고서 제목: {topic}
- 보고서 목적: {purpose}
- 보고서 요구사항: {requirements}
- 이전 내용 요약: {summary_context}

[출력 요구사항]
1. REPORT
- “해당 소제목”에 정확히 대응하는 심층적이고 장문의 본문을 작성하라.
- 단순 설명을 지양하고, 원인·구조·배경·영향·시사점의 관점에서 심층적이고 전문적인 분석을 수행하라.
- 해당 주제에 대해 객관적 사실에 기반한 정확한 분석을 제공하라.
- 관련 데이터(통계, 조사 결과, 시장 동향)와 사례(기존 연구, 산업 사례, 국가·지역 비교)를 적극 활용하여 논리를 강화하라.
- 이전 내용 요약을 참고하여 논리적 흐름이 자연스럽게 이어지도록 작성하라.
- 검증된 사실 범위 내에서 가능한 한 많은 유의미한 정보를 포함하라.
- 내용은 보고서 목적에 직접적으로 부합하도록 구성하라.
- 장 제목과 소제목은 “1. 서론”, “1.1 정의”와 같은 형태로만 작성하며, ##, ### 등 마크다운 헤더 기호는 절대 사용하지 마라.
- 문자 * 및 강조를 위한 모든 기호는 절대 사용하지 마라.
- 모든 본문은 빈 줄 없이 작성하라.
- 문단은 필요한 경우에만 구분할 수 있으며, 문단이 바뀔 때에는 단일 줄바꿈만 허용한다.
- 문단 내부의 일반 서술 문장 사이에는 줄바꿈을 사용하지 마라.
- 나열이 필요한 경우에만 줄바꿈을 허용하며,
  각 항목은 반드시 -기호로 시작하고,
  항목 간에는 단일 줄바꿈만 사용하라.
- 연속된 줄바꿈(빈 줄)은 절대 사용하지 마라.
- 소제목 바로 아래에도 빈 줄을 두지 마라.

2. SUMMARY
- 해당 소제목의 핵심 내용을 정확히 2문장으로 요약하라.
- 분석 결과와 시사점이 포함되도록 하라.

3. SOURCES
- 해당 장의 본문, 하위 질문, 분석 및 답변을 작성하는 과정에서
  사실 확인, 수치 인용, 분석 틀 설정, 판단 근거로 실제 사용된 모든 외부 출처를
  MLA 형식으로 기재하라.
- 직접 인용 여부와 무관하게 분석에 활용되었으면 포함하라.
- 중복 출처는 1회만 기재하라.

[출력 형식]
반드시 아래 3개 블록만으로 출력하라. 블록의 순서, 대괄호 태그 표기, 시작/종료 태그는 그대로 유지하라.

[REPORT]
(해당 소제목 보고서 본문)
[/REPORT]

[SUMMARY]
(해당 소제목 핵심 요약 2문장)
[/SUMMARY]

[SOURCES]
(MLA 형식 출처 목록)
[/SOURCES]
""".strip()

def parse_response_blocks(text: str, chapter_title: str = ""):
    t = (text or "").strip()
    if not t:
        return f"{chapter_title} 오류!", "", ""

    def between_pair(t: str, a: str, b: str) -> str:
        if a not in t or b not in t:
            return ""
        return t.split(a, 1)[1].split(b, 1)[0].strip()

    report = between_pair(t, "[REPORT]", "[/REPORT]")
    summary = between_pair(t, "[SUMMARY]", "[/SUMMARY]")
    sources = between_pair(t, "[SOURCES]", "[/SOURCES]")

    if report:
        return report, summary, sources

    if "[REPORT]" in t and "[SUMMARY]" in t and "[SOURCES]" in t:
        try:
            after_report = t.split("[REPORT]", 1)[1]
            report2 = after_report.split("[SUMMARY]", 1)[0].strip()

            after_summary = after_report.split("[SUMMARY]", 1)[1]
            summary2 = after_summary.split("[SOURCES]", 1)[0].strip()

            sources2 = after_summary.split("[SOURCES]", 1)[1].strip()

            if report2:
                return report2, summary2, sources2
        except Exception:
            pass

    return f"{chapter_title} 오류!", "", ""

def safe_extract_blocks(raw_text: str, chapter_title: str):
    return parse_response_blocks(raw_text, chapter_title)

# =========================
# 3) 핵심: 생성할 때마다 doc에 추가, 마지막에 save 1회
#    - 제목/TOC/Report/Sources 섹션 구성은 기존과 동일
# =========================
def build_docx_while_generating(
    topic: str,
    purpose: str,
    requirements: str,
    final_toc: str,
    api_key: str,
    summary_context: str = "",
    model: str = "gemini-2.0-flash",
    dedup_sources: bool = True,
    sleep_sec: float = 2.0,
):
    client = genai.Client(api_key=api_key)
    chapters = extract_chapters(final_toc)

    # (A) 문서 객체 생성 + 상단 구성(기존과 동일)
    doc = Document()
    doc.add_heading(topic, level=1)

    doc.add_heading("TOC", level=1)
    add_multiline_text(doc, final_toc)

    doc.add_heading("Report", level=1)

    updated_summary_context = summary_context or ""
    seen_sources = set()
    sources_out = []  # report는 doc에 바로 기록, sources만 최소로 유지

    for chapter_title in chapters:
        prompt = make_prompt_for_report(
            chapter_title=chapter_title,
            topic=topic,
            purpose=purpose,
            requirements=requirements,
            summary_context=updated_summary_context,
        )

        response = client.models.generate_content(model=model, contents=prompt)
        time.sleep(sleep_sec)

        raw_text = getattr(response, "text", "") or ""
        r, s, src = safe_extract_blocks(raw_text, chapter_title)

        # (B) 생성 결과를 즉시 doc에 기록
        doc.add_heading(chapter_title, level=2)
        add_multiline_text(doc, r or f"{chapter_title} 오류!")

        # SUMMARY는 다음 호출 입력용으로만 누적
        if s:
            updated_summary_context += f"- {chapter_title}: {s}\n"
        else:
            updated_summary_context += f"- {chapter_title}: (요약 없음)\n"

        # SOURCES는 마지막 섹션에서 일괄 출력
        if src:
            if dedup_sources:
                for line in src.splitlines():
                    item = line.strip()
                    if not item:
                        continue
                    if item not in seen_sources:
                        seen_sources.add(item)
                        sources_out.append(item)
            else:
                sources_out.append(src.strip())

    # (C) 문서 하단 Sources 섹션(기존과 동일)
    doc.add_heading("Sources", level=1)
    add_multiline_text(doc, "\n".join(sources_out).strip())

    # (D) 마지막에 save 1회
    file_name = f"{sanitize_filename(topic)}.docx"
    file_path = get_download_path(file_name)
    doc.save(file_path)

    return file_path, updated_summary_context

# =========================
# 4) 실행(바로 실행되는 구간)
#    - 아래 변수들은 기존 노트북/코드에서 이미 존재한다고 가정:
#      get_inputs(), toc_state["final_toc"], gemini_key
# =========================
topic, purpose, requirements = get_inputs()
final_toc = (toc_state.get("final_toc") or "").strip()

file_path, updated_summary_context = build_docx_while_generating(
    topic=topic,
    purpose=purpose,
    requirements=requirements,
    final_toc=final_toc,
    api_key=gemini_key,
    summary_context=updated_summary_context if "updated_summary_context" in globals() else "",
    model="gemini-2.0-flash",
    dedup_sources=True,
    sleep_sec=2.0,
)

print(f"파일이 저장되었습니다: {file_path}")

파일이 저장되었습니다: C:\Users\mphk0\Downloads\스페이스 엑스 관련주.docx
